# 07 — Paper Ranking

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src import config, semantic_search, ranking

print("Current ranking weights:", config.RANKING_WEIGHTS)


Current ranking weights: {'semantic': 0.65, 'citation': 0.15, 'recency': 0.1, 'quality': 0.1}


## Rank a set of search results

In [2]:
results = semantic_search.search_papers("deep learning for healthcare", top_k=15)
ranked = ranking.compute_scores(results)
pd.DataFrame(ranked)[["rank", "title", "final_score", "semantic_score", "citation_score", "recency_score", "quality_score"]]


2026-09-01 22:37:29,496 | INFO     | src.vector_store | Loaded faiss-backed index with 583 vectors from E:\Job Base Programe\ResearchMind\vector_db\faiss_index\index.faiss
2026-09-01 22:39:39,396 | INFO     | src.embeddings | Loading sentence-transformers model 'all-mpnet-base-v2' (this may download weights on first use)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,rank,title,final_score,semantic_score,citation_score,recency_score,quality_score
0,1,MONAI: An open-source framework for deep learn...,0.7469,0.7045,0.7888,0.7071,1.0
1,2,Deep Learning: A Comprehensive Overview on Tec...,0.7449,0.6895,0.8793,0.6484,1.0
2,3,Medical image analysis using deep learning alg...,0.7437,0.7092,0.7038,0.7711,1.0
3,4,Deep learning-enabled medical computer vision,0.7216,0.6707,0.8057,0.6484,1.0
4,5,Understanding of Machine Learning with Deep Le...,0.7106,0.6411,0.7786,0.7711,1.0
5,6,"Review of deep learning: concepts, CNN archite...",0.7030,0.5971,1.0000,0.6484,1.0
6,7,Deep learning modelling techniques: current pr...,0.6993,0.6247,0.7743,0.7711,1.0
7,8,Foundation models for generalist medical artif...,0.6952,0.5978,0.8634,0.7711,1.0
8,9,Survey of Explainable AI Techniques in Healthcare,0.6901,0.6254,0.7095,0.7711,1.0
9,10,Deep learning for lungs cancer detection: a re...,0.6873,0.6399,0.5821,0.8409,1.0


## Compare against a semantic-only baseline

Ranking must not let citation count dominate semantic relevance.

In [3]:
baseline = sorted(results, key=lambda r: r["similarity_score"], reverse=True)
weighted_top5 = [r["title"] for r in ranked[:5]]
baseline_top5 = [r["title"] for r in baseline[:5]]
overlap = len(set(weighted_top5) & set(baseline_top5))
print(f"Overlap between weighted ranking and semantic-only baseline (top 5): {overlap}/5")


Overlap between weighted ranking and semantic-only baseline (top 5): 5/5


## (Optional) Learned ranking model

Only trained if real labeled relevance data is supplied — ResearchMind does not fabricate a fake model.

In [4]:
try:
    ranking.load_learned_ranker()
except ranking.LearnedRankerUnavailable as e:
    print(e)


No learned ranker found at E:\Job Base Programe\ResearchMind\models\ranking\paper_ranker.pkl. Falling back to the transparent weighted-scoring approach (compute_scores()).
